# Galileo's Falling Body Experiment

**A note on this document**
This document is known as a Jupyter notebook; it allows text and executable code to coexist in a very easy-to-read format. Blocks can contain text or executable code. For blocks containing code, press `Shift + Enter`, `Ctrl + Enter`, or click the arrow on the block to run the code. Earlier blocks of code need to be run for the later blocks of code to work.

A ball is thrown upward from height $y_0$ with initial velocity $v_0$. Its height is measured at times $t_1, t_2, \cdots, t_n$, giving the measurements $y_1, y_2, \cdots, y_n$. From physics, the height follows

$$y(t) = \tfrac{1}{2} g t^2 + v_0 t + y_0,$$

so we model the data with a quadratic $\hat{y}(t) = at^2 + bt + c$. We want to find (estimate) the parameters $\mathbf{q} = [a \quad b \quad c]^\top$ that minimize $\|\mathbf{y} - \hat{\mathbf{y}}\|^2$.

This notebook solves the problem three ways: `Polynomial.fit`, the pseudo-inverse, and `np.linalg.lstsq`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from numpy.polynomial import Polynomial  # the Polynomial class from NumPy's polynomial package

# True parameters used to generate the data (we compare against them at the end)
y0 = 60  # initial height, m
v0 = 20  # initial velocity, m/s
g = -9.8067  # gravitational acceleration, m/s^2

data = pd.read_csv("./data/falling_body.csv")
data

Convert the DataFrame columns into NumPy arrays:

In [ ]:
t = data["time"].to_numpy()
y = data["y"].to_numpy()
print(t)
print(y)

Plot the data.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(t, y, "o")
ax.set_xlabel("time (s)")
ax.set_ylabel("height, y (m)")
ax.grid(True)
plt.show()

We will plot several fitted curves, so let's define a helper function that plots the measurements together with a fit.

In [ ]:
def plot_fit(t, y, y_hat, label):
    """Plot the measurements y and the fitted values y_hat versus time t."""
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(t, y, "o", label="measurements")
    ax.plot(t, y_hat, "-", label=label)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("height, y (m)")
    ax.grid(True)
    ax.legend()
    plt.show()

## Method 1: `Polynomial.fit`

NumPy's `Polynomial.fit` finds the least-squares polynomial of a given degree. Note that it always includes a constant term, so it generally can't handle a model whose intercept must be 0, such as $\hat{y} = at$.

Let's start with a straight line, $\hat{y} = at + b$, i.e., a first-degree polynomial.

In [ ]:
p1_scaled = Polynomial.fit(t, y, deg=1)  # fit the line using a rescaled version of t
p1 = p1_scaled.convert()  # rewrite the same line in terms of t itself

print("coefficients for the rescaled t:", p1_scaled.coef)
print("coefficients for t:", p1.coef)  # [b, a]: intercept, then slope

**What does `.convert()` do?** Before fitting, `Polynomial.fit` rescales the times so they lie between −1 and 1. Computers make smaller round-off errors with small numbers, which matters for high powers such as $t^4$. The coefficients in `p1_scaled` describe the line in terms of this rescaled time, which is why they don't match the slope and intercept of our data. `.convert()` rewrites the same line in terms of the original $t$, so `p1.coef` holds the $b$ and $a$ we want. Always call `.convert()` before reading the coefficients.

**Coefficient order.** `p1.coef` lists the coefficients from the lowest power up: `coef[0]` is the constant term and `coef[k]` multiplies $t^k$. For a line, `p1.coef` is $[b, a]$.

A `Polynomial` object is callable, so `p1(t)` evaluates the fitted line at every time in `t`.

In [ ]:
y_hat = p1(t)
plot_fit(t, y, y_hat, "degree 1")

The line doesn't fit the measurements well. To quantify this, look at the residuals (errors) $r_i = y_i - \hat{y}_i$ and the root mean squared error (RMSE):

$$\mathrm{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}$$

In linear algebra, the (Euclidean) norm of a vector $\mathbf{r} \in \mathbb{R}^n$ is

$$\|\mathbf{r}\| = \sqrt{\sum_{i=1}^{n} r_i^2},$$

so the RMSE can be written compactly as

$$\mathrm{RMSE} = \frac{1}{\sqrt{n}}\|\mathbf{y} - \hat{\mathbf{y}}\|.$$

Here is how we can find the RMSE:

In [ ]:
rmse = np.linalg.norm(y - y_hat) / np.sqrt(len(t))
print(f"RMSE (degree 1): {rmse:.3f} m")

Now try a quadratic model, $\hat{y} = at^2 + bt + c$.

In [ ]:
p2 = Polynomial.fit(t, y, deg=2).convert()  # fit, then rewrite in terms of t
print(p2.coef)  # [c, b, a]

In [ ]:
y_hat = p2(t)
plot_fit(t, y, y_hat, "degree 2")

rmse = np.linalg.norm(y - y_hat) / np.sqrt(len(t))
print(f"RMSE (degree 2): {rmse:.3f} m")

The quadratic model has a much lower RMSE than the line. Now try a fourth-degree polynomial.

In [ ]:
p4 = Polynomial.fit(t, y, deg=4).convert()  # fit, then rewrite in terms of t
print(p4.coef)

In [ ]:
y_hat = p4(t)
plot_fit(t, y, y_hat, "degree 4")

rmse = np.linalg.norm(y - y_hat) / np.sqrt(len(t))
print(f"RMSE (degree 4): {rmse:.3f} m")

The fourth-degree polynomial has a slightly lower RMSE than the quadratic, but that doesn't make it a better model. The RMSE on the same data used for fitting (the *training error*) can never increase as the degree goes up, because a higher-degree polynomial includes all lower-degree ones as special cases. The extra terms are fitting the noise, not the physics. This is called **overfitting**, and we'll study it in more depth later.

## Method 2: the pseudo-inverse

Writing the quadratic model at every measurement time gives a system of linear equations:

$$\begin{bmatrix} y_1 \\ y_2 \\ \vdots \\ y_n \end{bmatrix} = \begin{bmatrix} t^2_1 & t_1 & 1 \\ t^2_2 & t_2 & 1 \\ \vdots & \vdots & \vdots \\ t^2_n & t_n & 1 \end{bmatrix} \begin{bmatrix} a \\ b \\ c \end{bmatrix}$$

or $\mathbf{y} = C\mathbf{q}$, where $\mathbf{y} = [y_1 \quad y_2 \quad \cdots \quad y_n]^\top$, $\mathbf{q} = [a \quad b \quad c]^\top$, and $C$ is the $n \times 3$ matrix above.

We can build $C$ with `np.column_stack()`, which stacks 1-D arrays side by side as columns. Make sure $C$ is an $n \times 3$ matrix, i.e., $C\in\mathbb{R}^{n\times 3}$.

In [ ]:
C = np.column_stack([t**2, t, np.ones_like(t)])
print(C.shape)
print(C)

With more equations than unknowns ($n > 3$), $\mathbf{y} = C\mathbf{q}$ generally has no exact solution. The least-squares solution is $\mathbf{q} = \tilde{C}\mathbf{y}$, where $\tilde{C}$ is the pseudo-inverse of $C$. When the columns of $C$ are linearly independent, $\tilde{C} = (C^{\top}C)^{-1}C^{\top}$.

In Python, `np.linalg.pinv()` computes $\tilde{C}$ using the singular value decomposition (SVD), which is more numerically stable than forming $(C^{\top}C)^{-1}$ directly. $\tilde{C}$ should be $3 \times n$.

In [ ]:
pinvC = np.linalg.pinv(C)
print(pinvC.shape)
print(pinvC)

We can now find $\mathbf{q} = [a \quad b \quad c]^\top$:

$$\mathbf{q} = \tilde{C}\mathbf{y} = (C^{\top}C)^{-1}C^{\top}\mathbf{y}$$

In [ ]:
q = pinvC @ y
print(q)

The values in `q` match the quadratic coefficients from `Polynomial.fit`, in reverse order: `q` is $[a, b, c]$ and `p2.coef` is $[c, b, a]$. Since $\hat{\mathbf{y}} = C\mathbf{q}$, we can compute the fitted values with a matrix product.

In [ ]:
print(np.allclose(q, p2.coef[::-1]))

y_hat = C @ q
plot_fit(t, y, y_hat, "pseudo-inverse")

## Method 3: `np.linalg.lstsq`

In practice, use `np.linalg.lstsq`, which solves the least-squares problem directly without forming the pseudo-inverse. It returns a tuple `(solution, residuals, rank, singular_values)`; the first element is $\mathbf{q}$.

In [ ]:
q, residuals, rank, sv = np.linalg.lstsq(C, y)
print(q)
print("rank of C:", rank)

## Physical interpretation

Comparing $\hat{y} = at^2 + bt + c$ with $y(t) = \tfrac{1}{2} g t^2 + v_0 t + y_0$ gives $g \approx 2a$, $v_0 \approx b$, and $y_0 \approx c$. The estimates should be close to the true values used to generate the (noisy) data.

In [ ]:
a, b, c = q
print(f"g  = 2a = {2 * a:8.3f} m/s^2  (true: {g})")
print(f"v0 = b  = {b:8.3f} m/s    (true: {v0})")
print(f"y0 = c  = {c:8.3f} m      (true: {y0})")

To see the full documentation of `np.linalg.lstsq`:

In [ ]:
help(np.linalg.lstsq)